# 03 — Cortex Agent

**Business Context:** HydraB has "no documentation anywhere" (Jaco). Business users like Kim (CIO) and Andy (COO) need answers about fleet status, delivery pipeline, and vehicle health — but can't write SQL. A Cortex Agent lets them ask natural language questions and get instant answers from the unified data.

**What this notebook does:**
1. Creates a **Semantic View** — a business-friendly data model that maps technical columns to human-readable concepts (e.g., `battery_soc` → "Battery State of Charge %")
2. Creates a **Cortex Agent** — an AI assistant that converts natural language questions to SQL using the Semantic View

**Pain point addressed:** Non-technical stakeholders can now ask questions like:
- "How many vehicles does each customer operate?"
- "Which buses have battery SOC below 20%?"
- "What's our delivery pipeline status?"

**Why this matters for HydraB:** The team discussed needing a Vehicle 360 view that anyone can query. The Agent makes the GOLD layer self-service — no SQL knowledge required.


## Set Session Context
Resolves per-user database and sets the warehouse + schema.

In [ ]:
-- Set context
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE SCHEMA GOLD;
USE WAREHOUSE HYDRAB_HOL_WH;

## Semantic View

The Semantic View defines a business-friendly model for Cortex Analyst.

In [ ]:
-- Create Semantic View using SQL DDL syntax
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE SCHEMA GOLD;

CREATE OR REPLACE SEMANTIC VIEW FLEET_SEMANTIC

  TABLES (
    vehicles AS DIM_VEHICLE
      PRIMARY KEY (vin)
      COMMENT = 'Vehicle master data',
    telemetry AS FCT_LATEST_TELEMETRY
      COMMENT = 'Latest telemetry readings per vehicle'
  )

  RELATIONSHIPS (
    telemetry_to_vehicles AS
      telemetry (vin) REFERENCES vehicles (vin)
  )

  DIMENSIONS (
    vehicles.dim_vin AS vin
      COMMENT = 'Vehicle Identification Number',
    vehicles.dim_model AS model
      COMMENT = 'Bus model name',
    vehicles.dim_status AS status
      COMMENT = 'Vehicle lifecycle status',
    vehicles.dim_delivery_status AS delivery_status
      COMMENT = 'Delivery pipeline status',
    telemetry.dim_customer AS customer_name
      COMMENT = 'Operating customer'
  )

  METRICS (
    vehicles.total_vehicles AS COUNT(DISTINCT vin)
      COMMENT = 'Total number of vehicles',
    vehicles.deployed_count AS COUNT_IF(status = 'Deployed')
      COMMENT = 'Deployed vehicles',
    telemetry.avg_battery_soc AS AVG(battery_soc)
      COMMENT = 'Average battery state of charge',
    telemetry.avg_speed AS AVG(speed_kmh)
      COMMENT = 'Average speed km/h',
    telemetry.low_battery_count AS COUNT_IF(battery_soc < 20)
      COMMENT = 'Vehicles with SOC below 20 percent'
  )

  COMMENT = 'HydraB fleet analytics - vehicles and telemetry';

## Cortex Agent

The Agent combines structured analytics (via Semantic View) with natural language Q&A.

In [ ]:
-- Create the Fleet Intelligence Agent
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE SCHEMA GOLD;

CREATE OR REPLACE AGENT FLEET_AGENT
  COMMENT = 'HydraB Fleet Intelligence Agent'
  FROM SPECIFICATION
  $$
  tools:
    - tool_spec:
        type: "cortex_analyst_text_to_sql"
        name: "FleetAnalyst"
        description: "Answers questions about the HydraB electric bus fleet using vehicle and telemetry data"

  tool_resources:
    FleetAnalyst:
      semantic_view: "GOLD.FLEET_SEMANTIC"

  instructions:
    response: "You are a fleet intelligence assistant for HydraB Power electric buses. Answer questions about vehicles, battery SOC, speed, location, and fleet composition."
    sample_questions:
      - question: "How many vehicles does each customer operate?"
      - question: "Which vehicles have battery SOC below 20%?"
      - question: "What is the average speed across the fleet?"
  $$;

## Gold Layer Complete!

You now have:
- Auto-refreshing Dynamic Tables in SILVER
- Star schema views in GOLD
- A Semantic View for natural language analytics
- A Cortex Agent you can ask questions

**Next:** Open `04_deploy_dashboard.ipynb` to deploy the React dashboard.